In [ ]:
# Core data handling library
import pandas as pd

# Standard library utilities
import os
import sys

# Model evaluation metric for classification tasks
from sklearn.metrics import classification_report  # Precision, recall, F1 summary

# LazyPredict utility to benchmark multiple classifiers quickly
from lazypredict.Supervised import LazyClassifier  # Automated model comparison

# SMOTE oversampling technique to handle class imbalance
from imblearn.over_sampling import SMOTE  # Synthetic Minority Over-sampling

# Filesystem path handling
from pathlib import Path

# Set project root directory (parent of current working directory)
root_dir = Path.cwd().parent

# Add project root to Python module search path to enable local package imports without installing the project as a site-package
sys.path.append(str(root_dir))

# Import custom helper utilities from utils/helpers.py
from utils.helpers import set_global_settings

In [ ]:
# Define the random seed for reproducibility across runs
# Reads RANDOM_STATE from environment variable if set, otherwise defaults to 100
# int(...) ensures the value is an integer even if read as string
RANDOM_STATE = int(os.getenv("RANDOM_STATE", 100))

# Apply project-wide visualization and formatting settings
# Configures matplotlib/seaborn styles, fonts, and plotting defaults to ensure consistency
set_global_settings()

In [ ]:
# Define the directory containing prepared modeling datasets
dataset_dir = root_dir / "datasets"

# Load the training dataset (features + target) created during preprocessing
# This dataset already includes encoded categorical variables and transformed numerical features, ready for model training
train_df = pd.read_csv(dataset_dir / "train_data.csv")

# Verify successful load and inspect schema.
train_df.head()

,person_age,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-0.64,0,0.33,2,0.08,3,-1.20,-0.86,1,0
1,-0.64,1,-1.13,1,-0.43,1,-1.20,0.50,1,0
2,-0.32,0,-1.21,1,0.08,1,0.11,-0.05,0,0
3,-0.32,1,-0.81,1,-1.11,1,-0.57,-1.20,0,0
4,-1.00,3,0.55,2,0.65,0,0.89,0.01,0,1


In [ ]:
# Load the validation dataset created during preprocessing
# This split is used for model selection and hyperparameter tuning, while remaining unseen by the training process.
val_df = pd.read_csv(dataset_dir / "validation_set.csv")

# Verify successful load and inspect structure.
val_df.head()

,person_age,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-1.00,3,-0.31,2,1.85,1,-1.20,1.66,0,1
1,-1.42,2,0.10,2,-1.49,0,0.56,1.41,1,0
2,-0.32,2,-1.80,1,-0.43,3,-0.69,0.57,1,0
3,-1.00,2,-0.61,3,-1.72,0,0.06,1.36,0,0
4,0.77,2,-0.16,2,-0.21,5,-0.00,2.24,1,0


In [ ]:
# Define the target variable column name used for classification.
target_variable = 'loan_status'

# Compute and display the normalized class distribution in the training set
# value_counts(normalize=True) → proportion of each class (not raw counts)
train_df[target_variable].value_counts(normalize=True)

loan_status
0   0.78
1   0.22
Name: proportion, dtype: float64

In [ ]:
# Separate features (X) and target (y) for the training dataset
X_train = train_df.drop(columns=[target_variable])
y_train = train_df[target_variable]

# Separate features (X) and target (y) for the validation dataset
# These will be used for model evaluation/tuning
X_val = val_df.drop(columns=[target_variable])
y_val = val_df[target_variable]

# Report dataset sizes for verification and experiment tracking.
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")

Training set size: 30732 samples
Validation set size: 8780 samples


In [ ]:
# Calculate class distribution (%) in the original training target
# value_counts(normalize=True) → proportions; *100 → convert to percentage
freq_y_train = y_train.value_counts(normalize=True) * 100

# Apply SMOTE to the training set to balance class distribution
# Only applied to training data to avoid leakage into validation/test sets
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Calculate class distribution (%) after SMOTE resampling
freq_y_train_sm = pd.Series(y_train_sm).value_counts(normalize=True) * 100

# Combine before/after distributions into a single comparison table
# axis=1 → align by class label as columns
# keys → column names in resulting DataFrame
# fillna(0) → handle any class absent in one distribution
# round(2) → format percentages to 2 decimal places
dist = pd.concat(
    [freq_y_train, freq_y_train_sm],
    axis=1,
    keys=['Before SMOTE (%)', 'After SMOTE (%)']
).fillna(0).round(2)

# Print formatted markdown table for readability.
print(f'Class distribution for {target_variable} (Train set)\n')
print(dist.to_markdown(), '\n')

Class distribution for loan_status (Train set)

|   loan_status |   Before SMOTE (%) |   After SMOTE (%) |
|--------------:|-------------------:|------------------:|
|             0 |              77.71 |                50 |
|             1 |              22.29 |                50 | 



In [ ]:
# Initialize LazyClassifier to benchmark multiple classification models with minimal configuration
# predictions=True → also return validation set predictions per model
# random_state → ensure reproducibility for models with randomness
clf = LazyClassifier(
    predictions=True,
    random_state=RANDOM_STATE
)

# Train and evaluate a suite of classifiers automatically.
# Returns:
    # models → DataFrame with performance metrics per model
    # predictions → validation predictions for each model
models, predictions = clf.fit(X_train_sm, X_val, y_train_sm, y_val)

# Sort model comparison table by key performance metrics (best-performing models first)
# Priority: F1 Score → Balanced Accuracy → ROC AUC → Time Taken
models.sort_values(
    by=['F1 Score', 'Balanced Accuracy', 'ROC AUC', 'Time Taken'],
    ascending=False
)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 23883, number of negative: 23883
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002168 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 47766, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
XGBClassifier,0.92,0.89,0.89,0.92,0.60
LGBMClassifier,0.91,0.89,0.89,0.91,7.31
RandomForestClassifier,0.91,0.89,0.89,0.91,22.05
ExtraTreesClassifier,0.91,0.89,0.89,0.91,5.20
BaggingClassifier,0.91,0.88,0.88,0.91,2.06
DecisionTreeClassifier,0.89,0.86,0.86,0.89,0.36
SVC,0.87,0.88,0.88,0.88,98.05
KNeighborsClassifier,0.87,0.87,0.87,0.87,2.18
AdaBoostClassifier,0.86,0.88,0.88,0.87,2.67


Top 5 models:

1. `XGBClassifier`

2. `LGBMClassifier`

3. `RandomForestClassifier`

4. `ExtraTreesClassifier`

5. `BaggingClassifier`

In [ ]:
# Define a shortlist of high-performing models selected from LazyClassifier benchmarking results for detailed evaluation
models_of_interest = [
    'XGBClassifier', 'LGBMClassifier', 'RandomForestClassifier',
    'ExtraTreesClassifier', 'BaggingClassifier'
]

# Iterate through each selected model and print a detailed classification report on the validation set
for model in models_of_interest:
    
    # Print model name as a section header for readability
    print('\t\t', model, '\n')
    
    # Display precision, recall, F1-score, and support per class using validation labels vs model predictions
    # predictions[model] → predicted labels from LazyClassifier output
    print(classification_report(y_val, predictions[model]), '\n')

		 XGBClassifier 

              precision    recall  f1-score   support

           0       0.96      0.94      0.95      6823
           1       0.79      0.85      0.82      1957

    accuracy                           0.92      8780
   macro avg       0.87      0.89      0.88      8780
weighted avg       0.92      0.92      0.92      8780
 

		 LGBMClassifier 

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6823
           1       0.78      0.85      0.81      1957

    accuracy                           0.91      8780
   macro avg       0.87      0.89      0.88      8780
weighted avg       0.92      0.91      0.91      8780
 

		 RandomForestClassifier 

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6823
           1       0.78      0.85      0.81      1957

    accuracy                           0.91      8780
   macro avg       0.87      0.89      0.88      8780
wei

In [ ]:
# Combine SMOTE-resampled training features and target labels into a single DataFrame for downstream modeling.
sampled_df = pd.DataFrame(X_train_sm, columns=X_train.columns)

# Append the resampled target variable as a column
sampled_df[target_variable] = y_train_sm

# Define output path for the SMOTE-balanced training dataset
file_path = dataset_dir / "sampled_train_data.csv"

# Save the resampled training dataset to CSV for reuse without recomputing SMOTE
sampled_df.to_csv(file_path, index=False)